In [78]:
import torch
from torch import nn
import pandas as pd
import random
import optuna
import numpy as np
from torch.optim.lr_scheduler import OneCycleLR

In [ ]:
#See num_seq2seq for what this dataframe contains
df = pd.read_csv("gru_reversal_data.csv")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)
print(len(df))

cuda
79994


In [80]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
print(device)

cuda


In [ ]:
#To treat them as lists, since pandas saves them as strings
df['source'] = df['source'].apply(lambda x: eval(x))
df['context_vector'] = df['context_vector'].apply(lambda x: eval(x))

In [ ]:
#Ensure reproducibility
seed_num = 4
random.seed(seed_num)
np.random.seed(seed_num)
torch.manual_seed(seed_num)
torch.cuda.manual_seed_all(seed_num)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
#Easier to do bucket batching
df_sorted = df.sort_values(by='source', key=lambda col: col.map(len))

In [ ]:
#Model size is what our final w0 projects to
model_size = len(df_sorted['context_vector'].iloc[0])
batch_size = 128
print(model_size)   

128


In [ ]:
#Functions to create left/right branching trees
left_role = 0
right_role = 1
def right_recurse(lst):
    if not lst:
        return []
    
    batch_size = lst[0].shape[0]
    tensors = [[x, torch.full((batch_size,), left_role, device = device)] for x in lst[:-1]]
    
    acc = lst[-1]
    for t in reversed(tensors):
        acc = [t, [acc, torch.full((batch_size,), right_role, device = device)]]
    return acc

def left_recurse(lst):
    if not lst:
        return []
    
    batch_size = lst[0].shape[0]
    tensors = [[x, torch.full((batch_size,), right_role, device = device)] for x in lst[1:]]
    
    acc = lst[0]
    for t in tensors:
        acc = [[acc, torch.full((batch_size,), left_role, device = device)], t]
    return acc

In [ ]:
#Stack lists and cast them to tensors
def collate_lists(*lists):
    return [torch.tensor([lst[i] for lst in lists]) for i in range(len(lists[0]))]

In [ ]:
sources = df_sorted["source"].tolist()
targets = df_sorted["context_vector"].tolist()

from collections import defaultdict

buckets = defaultdict(list) 

for src, trg in zip(sources, targets):
    buckets[len(src)].append((src, trg))

#To test longer sequences as inputs to the TPDN later
del buckets[9]
del buckets[10]
del buckets[11]

In [ ]:
src_batches = []
trg_batches = []

#Bucket batching and 80/10/10 Train/Test/Val split

for seq_len in sorted(buckets.keys()):
    pairs = buckets[seq_len]
    for i in range(0, len(pairs), batch_size):
        # If i + batch_size is out of bounds, it just goes to the end of the list
        chunk = pairs[i:i + batch_size]
        src_batch = [p[0] for p in chunk]
        trg_batch = [p[1] for p in chunk]

        collated = collate_lists(*src_batch) 
        # nested   = right_recurse(collated)
        nested = left_recurse(collated)

        src_batches.append(nested)
        trg_batches.append(torch.stack([torch.tensor(v, dtype=torch.float32) for v in trg_batch]))

n = len(src_batches)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

combined = list(zip(src_batches, trg_batches))
random.shuffle(combined)
src_batches, trg_batches = zip(*combined)

train_src, train_trg = src_batches[:n_train], trg_batches[:n_train]
val_src, val_trg = src_batches[n_train:n_train+n_val], trg_batches[n_train:n_train+n_val]
test_src, test_trg = src_batches[n_train+n_val:], trg_batches[n_train+n_val:]

In [194]:
def get_recursive_batch_size(x):
    if isinstance(x[0], list):
        return get_recursive_batch_size(x[0])
    return x[0].shape[0]
    
class RTPREncode(nn.Module):
    def __init__(self, nfillers, nroles, filler_size, role_size, model_size):
        super(RTPREncode, self).__init__()
        self.fillers = nn.Embedding(nfillers, filler_size)
        self.roles = nn.Embedding(nroles, role_size)
        self.filler_size = filler_size
        self.role_size = role_size
        self.model_size = model_size
        # self.dropout = nn.Dropout(0.4)
        self.w = torch.nn.Parameter(torch.randn(filler_size, filler_size*role_size)*0.01)
        self.wo = torch.nn.Parameter(torch.randn(model_size, filler_size)*0.01)

    def forward(self, x):
        cur_batch_size = get_recursive_batch_size(x)
        
        result = torch.zeros(cur_batch_size, self.filler_size, self.role_size, device=self.w.device)
    
        for pair in x:
            filler = pair[0]
            role = pair[1].to(device)
            
            if isinstance(filler, list):
                filler = self(filler)
            else:
                filler = self.fillers(filler.to(device))
            
            role = self.roles(role)
            result += torch.einsum("bf,br->bfr", filler, role)
        
        result = result.view(cur_batch_size, -1)
        result = torch.einsum("ij,bj->bi", self.w, result) 
        return result

In [195]:
#Evaluate a model (for validation and testing loss)
def evaluate(model, src, trg, criterion, device):
    with torch.no_grad():
        model.eval()
        epoch_loss = 0.0

        for i in range(len(src)):
            src_batch = src[i]
            trg_batch = trg[i].to(device)
            encoding = model(src_batch)
            output = torch.matmul(encoding, model.wo.T)    # (batch_size, model_size)
            loss = criterion(output, trg_batch)
            epoch_loss += loss.item()

        return epoch_loss / len(src)

In [ ]:
#To allow Optuna to fine-tune
def objective(trial):
    random.seed(seed_num)
    np.random.seed(seed_num)
    torch.manual_seed(seed_num)
    torch.cuda.manual_seed_all(seed_num)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    num_epochs = 50

    model = RTPREncode(12, 2, 64, 64, model_size).to(device)
    learning_rate = trial.suggest_float("lr", 1e-5, 1e-3, log = True)
    norm_val = trial.suggest_float("grad_norm", 1, 10, log = True)
    max_rate = trial.suggest_float("scheduler_max", 1e-4, 1e-2, log = True)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    criterion = nn.MSELoss()
    scheduler = OneCycleLR(optimizer, max_lr=max_rate, steps_per_epoch=len(train_src), epochs=num_epochs)

    for epoch in range(num_epochs):
        # print(f"Epoch {epoch+1} / {num_epochs}")
        # print(f"Trial {trial.number}")
        model.train()

        epoch_loss = 0.0
        with torch.set_grad_enabled(True):
            for i in range(len(train_src)):
                src_batch = train_src[i]
                trg_batch = train_trg[i].to(device)
                optimizer.zero_grad()

                encoding = model(src_batch)
                output = torch.matmul(encoding, model.wo.T)

                loss = criterion(output, trg_batch)
                loss.backward()

                #Norm=magnitude, clipping gradient to ensure they don't explode
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = norm_val)
                optimizer.step()
                scheduler.step()
                epoch_loss += loss.item()

        # print(f"Train Loss: {epoch_loss/len(train_src)}")
        val_loss = evaluate(model, val_src, val_trg, criterion, device)

        trial.report(val_loss, epoch)
        #prune cuts off trial early
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        # print(f"Validation Loss: {val_loss}")

    return val_loss
    

In [197]:
from optuna.samplers import TPESampler

sampler = TPESampler(seed=seed_num)

study = optuna.create_study(sampler=sampler, direction='minimize')
study.optimize(objective, n_trials=8)
print("Best Hyperparameters:", study.best_params) 

[I 2026-07-14 18:06:21,463] A new study created in memory with name: no-name-c5aeab27-de25-4fbb-9625-89ee8e2f0623
[I 2026-07-14 18:07:04,274] Trial 0 finished with value: 0.0014941158632819469 and parameters: {'lr': 0.0008591315698733484, 'grad_norm': 3.52559360037893, 'scheduler_max': 0.008817972090698543}. Best is trial 0 with value: 0.0014941158632819469.
[I 2026-07-14 18:07:47,138] Trial 1 finished with value: 0.0014524414168240933 and parameters: {'lr': 0.00026892550160276973, 'grad_norm': 4.985730790035687, 'scheduler_max': 0.0002705073009621015}. Best is trial 1 with value: 0.0014524414168240933.
[I 2026-07-14 18:08:30,466] Trial 2 finished with value: 0.0014504328211888862 and parameters: {'lr': 0.0008964971423446301, 'grad_norm': 1.014449086034439, 'scheduler_max': 0.00032060089083377}. Best is trial 2 with value: 0.0014504328211888862.
[I 2026-07-14 18:09:14,370] Trial 3 finished with value: 0.0014526471928454554 and parameters: {'lr': 7.405989036592357e-05, 'grad_norm': 6.01

Best Hyperparameters: {'lr': 0.0008964971423446301, 'grad_norm': 1.014449086034439, 'scheduler_max': 0.00032060089083377}


In [198]:
pruned_trials = study.get_trials(deepcopy=False, states=[optuna.trial.TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[optuna.trial.TrialState.COMPLETE])

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

Study statistics: 
  Number of finished trials:  8
  Number of pruned trials:  3
  Number of complete trials:  5
Best trial:
  Value:  0.0014504328211888862
  Params: 
    lr: 0.0008964971423446301
    grad_norm: 1.014449086034439
    scheduler_max: 0.00032060089083377


In [199]:
# Train with optimized hyperparams
random.seed(seed_num)
np.random.seed(seed_num)
torch.manual_seed(seed_num)
torch.cuda.manual_seed_all(seed_num)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

num_epochs = 50

model = RTPREncode(12, 2, 64, 64, model_size).to(device)
learning_rate = 0.0008964971423446301
norm_val = 1.014449086034439
max_rate = 0.00032060089083377

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
criterion = nn.MSELoss()
scheduler = OneCycleLR(optimizer, max_lr=max_rate, steps_per_epoch=len(train_src), epochs=num_epochs)

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1} / {num_epochs}")
    # print(f"Trial {trial.number}")
    model.train()

    epoch_loss = 0.0
    with torch.set_grad_enabled(True):
        for i in range(len(train_src)):
            src_batch = train_src[i]
            trg_batch = train_trg[i].to(device)
            optimizer.zero_grad()

            encoding = model(src_batch)
            output = torch.matmul(encoding, model.wo.T)

            loss = criterion(output, trg_batch)
            loss.backward()

            #Norm=magnitude, clipping gradient to ensure they don't explode
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = norm_val)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

    print(f"Train Loss: {epoch_loss/len(train_src)}")
    val_loss = evaluate(model, val_src, val_trg, criterion, device)
    print(f"Validation Loss: {val_loss}")

Epoch 1 / 50
Train Loss: 0.1101004144022609
Validation Loss: 0.04610174684188305
Epoch 2 / 50
Train Loss: 0.03860942914987066
Validation Loss: 0.03265608011338955
Epoch 3 / 50
Train Loss: 0.024699251514497648
Validation Loss: 0.017584700376177445
Epoch 4 / 50
Train Loss: 0.01376769216553231
Validation Loss: 0.009566342947670283
Epoch 5 / 50
Train Loss: 0.008371252141697155
Validation Loss: 0.005956538982737141
Epoch 6 / 50
Train Loss: 0.005746993587714187
Validation Loss: 0.007326929465843699
Epoch 7 / 50
Train Loss: 0.004606402230339279
Validation Loss: 0.0029496349400291457
Epoch 8 / 50
Train Loss: 0.00489258716160218
Validation Loss: 0.0030921801733664977
Epoch 9 / 50
Train Loss: 0.004812750139327091
Validation Loss: 0.003283173935368466
Epoch 10 / 50
Train Loss: 0.005347248435592884
Validation Loss: 0.0028145919339014934
Epoch 11 / 50
Train Loss: 0.004810527022475686
Validation Loss: 0.004368121664111431
Epoch 12 / 50
Train Loss: 0.004829335350848758
Validation Loss: 0.002586917886

In [200]:
print(evaluate(model, test_src, test_trg, criterion, device))

0.001392645132727921


In [201]:
torch.save(model.state_dict(), "left_reversals/fixed_gru_left_reversal_4.pt")

In [76]:
state_dict = torch.load("left_copies/fixed_gru_left_copy_3.pt", weights_only=True)
model.load_state_dict(state_dict)

<All keys matched successfully>